# Notebook 01 experiment orchestration
Purpose: versioned configuration, predecessor evidence checks and reusable orchestration for the smoke and explicitly selected pilot modes. Inputs: project root, pinned data/configuration, Notebook00 handoff. Outputs: unique artifact directories and actual certificates. Definitions import without running experiments. Expected setup/test runtime <1 minute plus dataset preparation; no checkpoint for this setup module. Experiment checkpoints live in `checkpoints/<run_id>-<kind>/`. Resume uses `FMM_JEPA_01_RESUME_RUN_ID` with identical profile/config/source/protocol; invalid state must fail closed.

Notebook-only packaging is preserved from the user's explicit preference. The entrypoint notebook imports shared functions rather than using `%run` or upstream experiment state.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
from pathlib import Path
import copy, datetime, hashlib, importlib.metadata, json, math, os, platform, sys, uuid
import numpy as np
import torch
from fmm_jepa.training import atomic_json, digest_file, definition_hashes, write_csv

WORKFLOW_VERSION='0.1.0+tiny01'
CONFIG_SCHEMA='fmm-jepa-tiny-config/v1'
ARTIFACT_SCHEMA='fmm-jepa-tiny-artifact/v1'

def identity_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def resolve_config(root, profile=None):
    profile=profile or os.environ.get('FMM_JEPA_01_PROFILE','smoke')
    names={'smoke':'tiny_imagenet_smoke.yaml','pilot':'tiny_imagenet_pilot.yaml','full_pool':'tiny_imagenet_full_pool.yaml'}
    if profile not in names:raise ValueError('Profile must be smoke, pilot or full_pool')
    config=json.loads((Path(root)/'configs'/names[profile]).read_text())
    if config['schema_version']!=CONFIG_SCHEMA:raise ValueError('Unknown config schema')
    if config['device']=='auto':config['device']='cuda:0' if torch.cuda.is_available() else 'cpu'
    if config['device']=='cpu' and config['run_mode']=='smoke':
        config['model'].update(dim=32,depth=1,heads=4,predictor_dim=32,predictor_depth=1,predictor_heads=4)
        config['training'].update(updates=4,microbatch=2,accumulation_steps=1)
        config['evaluation'].update(batch_size=16,diagnostic_steps=[0,2,4])
        config['cpu_fallback']='Reduced width/depth/updates explicitly; same real datasets and four controls'
    if config['run_mode']=='smoke' and config['evaluation']['heldout_enabled']:raise ValueError('Smoke cannot access held-out evaluation')
    if config['world_size']!=1:raise NotImplementedError('Notebook01 supports one process/device only')
    if config['dtype']!='float32':raise ValueError('Mixed precision requires separate operator dtype certificates')
    config['effective_batch_size']=config['training']['microbatch']*config['training']['accumulation_steps']*config['world_size']
    torch.set_num_threads(config['cpu_threads'])
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    return config

def start_run(root, config, resume_run_id=None):
    root=Path(root)
    run_id=resume_run_id or datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
    if Path(run_id).name!=run_id:raise ValueError('Run ID must be one safe path component')
    artifact=root/'artifacts'/'notebook_01'/run_id
    if resume_run_id:
        if not (artifact/'resolved_config.json').is_file():raise FileNotFoundError('Resume run missing resolved config')
        old=json.loads((artifact/'resolved_config.json').read_text())
        if old!=config:raise ValueError('Config changed; register a new run rather than overwriting')
    else:artifact.mkdir(parents=True,exist_ok=False)
    for name in ('features','predictions','figures','checkpoints'): (artifact/name).mkdir(exist_ok=True)
    atomic_json(artifact/'resolved_config.json',config)
    return run_id,artifact

def verify_predecessor(root,artifact):
    from fmm_tests.test_geometry import run_geometry_tests
    from fmm_tests.test_models import run_model_tests
    from fmm_jepa.training import checkpoint_resume_test
    root=Path(root);artifact=Path(artifact)
    handoff=json.loads((root/'handoff_00_to_01.json').read_text())
    if handoff['schema_version']!='fmm-jepa-artifact/v1' or handoff['config_schema_version']!='fmm-jepa-config/v1':
        raise ValueError('Incompatible Notebook00 schemas')
    if not handoff['ready_for_notebook_01']:raise ValueError('Predecessor did not close its required gates')
    for name,expected in handoff['source_hashes'].items():
        doc=json.loads((root/name).read_text())
        source='\n'.join(''.join(c['source']) for c in doc['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[]))
        if hashlib.sha256(source.encode()).hexdigest()!=expected:raise ValueError(f'Predecessor exported source changed: {name}')
    previous=Path(handoff['artifact_directory']);clean=json.loads((previous/'clean_execution.json').read_text())
    if clean['status']!='PASS':raise ValueError('Predecessor clean execution was not verified')
    original=json.loads((root/'notebooks/00_FMM_JEPA_Core_Theory_and_Verification.ipynb').read_text())
    if any(o.get('output_type')=='error' for c in original['cells'] for o in c.get('outputs',[])):raise ValueError('Notebook00 contains an error output')
    if not all(g['status']=='PASS' for g in handoff['required_gates']):raise ValueError('Predecessor required gate is unclosed')
    # Additional Notebook01 modules do not alter the predecessor hash subset.
    results={**run_geometry_tests(),**run_model_tests()}
    core_config=json.loads((root/'configs/core_cpu_smoke.yaml').read_text())
    results.update(checkpoint_resume_test(root/'checkpoints',core_config,handoff['source_hashes']))
    report={'status':'PASS' if all(v['status']=='PASS' for v in results.values()) else 'FAIL',
            'predecessor_run_id':handoff['run_id'],'predecessor_artifact_directory':str(previous),
            'source_hashes_verified':handoff['source_hashes'],'clean_execution':clean,'fresh_certificates':results}
    atomic_json(artifact/'predecessor_verification.json',report)
    if report['status']!='PASS':raise RuntimeError('Essential predecessor core failed reverification; integrated training blocked')
    return report

def run_environment(config):
    import psutil, subprocess
    packages={name:importlib.metadata.version(name) for name in ['torch','numpy','Pillow','pyarrow','nbformat','nbclient']}
    mount=subprocess.run(['findmnt','-T',str(Path(__file__).parent.parent)],capture_output=True,text=True)
    return {'python':sys.version,'packages':packages,'device':config['device'],'dtype':config['dtype'],
            'gpu_names':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            'cuda_version':torch.version.cuda,'available_ram_bytes':psutil.virtual_memory().available,
            'cuda_free_total_bytes':list(torch.cuda.mem_get_info(config['device'])) if str(config['device']).startswith('cuda') else None,
            'cpu_threads':torch.get_num_threads(),'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),
            'microbatch':config['training']['microbatch'],'accumulation_steps':config['training']['accumulation_steps'],
            'effective_batch_size':config['effective_batch_size'],'world_size':1,'storage_mount':mount.stdout.strip(),
            'storage_persistence':'persistent boot disk; instance-deletion policy unverified; off-instance export required before recycle'}


In [ ]:

def save_dataset_evidence(bundle,artifact):
    for key in ('dataset_manifest','split_manifest'):
        atomic_json(Path(artifact)/(key+'.json'),bundle[key])
    atomic_json(Path(artifact)/'duplicate_audit.json',bundle['audit'])
    return {'dataset_fingerprint':bundle['dataset'].fingerprint,
            'selected_sizes':bundle['split_manifest']['selected_sizes'],
            'pool_sizes':bundle['split_manifest']['pool_sizes'],
            'split_sha256':bundle['split_manifest']['split_sha256']}

def geometry_evidence(config,batch,artifact):
    from fmm_jepa.geometry import build_geometry
    g=build_geometry((8,8),config['model']['order'],config['model']['leaf_size'])
    masks=batch['masks'];visible=masks['context_indices'][0][masks['context_valid'][0]]
    active=g.point_leaf[visible]
    occupied={}
    for level in range(g.depth,-1,-1):
        occupied[str(level)]=int(active.unique().numel())
        if level:active=g.parents[level][active]
    result={'grid':[8,8],'n_grid':64,'order':g.order,'nodes_per_cell':g.order**2,
            'leaf_size':g.leaf_size,'neighborhood_radius_cells':1,'hierarchy_depth':g.depth,
            'allocated_cells_per_level':[4**level for level in range(g.depth+1)],
            'occupied_cells_per_level_first_context':occupied,
            'geometry_stats':g.stats,'fullgrid_near_ordered_pairs':int(g.near_sources.numel()),
            'far_cell_interactions_per_level':[int(x.numel()) for x in g.m2l_sources],
            'visible_counts':masks['context_valid'].sum(1).tolist(),
            'target_counts':[x.sum(1).tolist() for x in masks['target_valid']],
            'mask_metadata':batch['mask_metadata'],
            'overhead_note':'p=4 means16 coefficients/cell/channel; only64 image tokens. Genuine far interactions do not imply speed advantage.'}
    assert sum(result['far_cell_interactions_per_level'])>0
    atomic_json(Path(artifact)/'geometry_evidence.json',result);return result

def register_protocol(root,artifact,run_id,config,bundle,counts,initialization,environment,predecessor):
    from fmm_jepa.tiny_training import training_protocol
    path=Path(artifact)/'protocol.json'
    value={'schema_version':ARTIFACT_SCHEMA,'run_id':run_id,'code_version':WORKFLOW_VERSION,
           'predecessor_run_id':predecessor['predecessor_run_id'],
           'dataset_repository':config['data']['repo_id'],'source_revision':config['data']['source_revision'],
           'parquet_revision':config['data']['parquet_revision'],'dataset_fingerprint':bundle['dataset'].fingerprint,
           'split_sha256':bundle['split_manifest']['split_sha256'],'model':config['model'],
           'parameter_counts':counts,'comparison':'same-scaffold; equal successful updates and paired data; not capacity/equal-compute matched',
           'initialization':initialization,'seeds':config['seeds'],'training':training_protocol(config),
           'preprocessing':config['preprocessing'],'masking':config['masking'],
           'paired_randomness':'Independent model-init, epoch-order, sampleID+occurrence augmentation, sampleID+occurrence mask and probe streams; num_workers=0.',
           'probe':config['probe'],'evaluation':config['evaluation'],'diagnostics':config['diagnostics'],
           'checkpoint_selection':config['checkpoint_selection'],'environment':environment,
           'source_hashes':definition_hashes(root),'test_source_hashes':{str(p.relative_to(root)):identity_hash('\n'.join(''.join(c['source']) for c in json.loads(p.read_text())['cells'] if 'ahr-export' in c.get('metadata',{}).get('tags',[]))) for p in sorted((Path(root)/'tests').glob('*.ipynb'))},'training_seed_count':1,
           'learning_rule':'Smoke always inconclusive. Pilot must complete both budgets, FMM trained-minus-initial top1 >0 with paired95%CI lower>0, and no unresolved zero-variance/low-rank flag. One seed remains preliminary.',
           'heldout_rule':'Disabled in smoke; explicit pilot flag only after final encoders and all selected probes fixed.',
           'resume_policy':'Exact config, dataset, protocol and export hashes required. Invalid manifests stop; no silent fresh start.',
           'stop_rules':'Nonfinite losses/gradients skip without EMA/schedule advancement; cap consecutive failures; nonfinite updated tensors stop at previous valid checkpoint; gate failure blocks comparison.'}
    if path.exists():
        old=json.loads(path.read_text())
        # Available RAM and wall-clock environment are observations, not changed protocol choices.
        volatile={'available_ram_bytes','cuda_free_total_bytes'}
        if {k:v for k,v in value['environment'].items() if k not in volatile}!={k:v for k,v in old['environment'].items() if k not in volatile}:
            raise ValueError('Stable software/hardware/determinism environment changed; register a new run')
        value['environment']=old['environment']
        if value!=old:raise ValueError('Registered protocol differs: use a new run ID with documented change')
        return old
    atomic_json(path,value);atomic_json(Path(artifact)/'parameter_counts.json',counts)
    atomic_json(Path(artifact)/'run_manifest.json',{'schema_version':ARTIFACT_SCHEMA,'run_id':run_id,'status':'REGISTERED','protocol_hash':identity_hash(value),'source_hashes':value['source_hashes'],'artifact_directory':str(artifact)})
    return value

def verification_suite(root,artifact,config,bundle,models):
    from fmm_tests.test_tiny_imagenet_pipeline import run_tiny_imagenet_pipeline_tests
    from fmm_tests.test_real_integration import run_real_integration_tests
    from fmm_tests.test_training_resume import run_real_resume_certificate,run_training_accounting_tests
    from fmm_tests.test_frozen_probe import run_frozen_probe_tests
    evidence={}
    evidence['data']=run_tiny_imagenet_pipeline_tests(bundle,config)
    evidence['integration']=run_real_integration_tests(models,bundle,config)
    evidence['accounting']=run_training_accounting_tests(config,bundle)
    evidence['resume']=run_real_resume_certificate(config,bundle,Path(artifact)/'resume',Path(root)/'checkpoints'/(Path(artifact).name+'-resume'),project_root=root)
    evidence['probe']=run_frozen_probe_tests(root)
    atomic_json(Path(artifact)/'verification_suite.json',evidence)
    def failed(value):
        if isinstance(value,dict):return value.get('status')=='FAIL' or any(failed(v) for v in value.values())
        if isinstance(value,list):return any(failed(v) for v in value)
        return False
    if failed(evidence):raise RuntimeError('A required certificate failed; see verification_suite.json')
    return evidence

def train_pair(root,artifact,run_id,config,bundle,models,protocol):
    from fmm_jepa.tiny_imagenet import EvaluationDataset
    from fmm_jepa.tiny_training import run_model
    from fmm_jepa.tiny_evaluation import compute_diagnostics
    selected=[];classes=set()
    for sid in bundle['dev_ids']:
        label=bundle['dataset'].get(sid)['label']
        if label not in classes:classes.add(label);selected.append(sid)
    assert len(classes)==200
    atomic_json(Path(artifact)/'diagnostic_membership.json',{'split':'development','sample_ids':selected,'count':len(selected),'selection':'first selected dev example per class; fixed before training'})
    diagnostic_data=EvaluationDataset(bundle['dataset'],selected,config)
    diagnostics=[]
    def callback(model,kind,step):
        value=compute_diagnostics(model.target,diagnostic_data,config)
        value.update(model_id=kind,seed=config['seeds']['model_init'],phase='representation_diagnostic',split='development',step=step)
        diagnostics.append(value);atomic_json(Path(artifact)/'representation_diagnostics.json',diagnostics)
        return value
    trained={}
    for kind in ('vit','fmm'):
        print(f'{kind}: starting/resuming registered {config["training"]["updates"]}-update budget',flush=True)
        trained[kind]=run_model(kind,config,bundle,artifact,Path(root)/'checkpoints'/(run_id+'-'+kind),protocol,model=models[kind],diagnostic_callback=callback)
        print(f'{kind}: {trained[kind]["resume"]}',flush=True)
    # Resume retains diagnostic history in authoritative checkpoints.
    diagnostics=[]
    for kind,value in trained.items():
        for row in value['diagnostics']:
            if isinstance(row,dict) and 'results' in row:row={**row['results'],'model_id':kind,'step':row.get('step')}
            diagnostics.append(row)
    metrics=[{**row,'run_id':run_id,'checkpoint_manifest':str(Path(v['checkpoint_dir'])/'manifest.json'),'state_reference':run_id+'/'+row['model_id']+'/successful_update_'+str(row['successful_updates'])} for v in trained.values() for row in v['metrics']]
    atomic_json(Path(artifact)/'representation_diagnostics.json',diagnostics)
    write_csv(Path(artifact)/'representation_diagnostics.csv',diagnostics)
    write_csv(Path(artifact)/'training_metrics.csv',metrics)
    traces={k:v['traces'] for k,v in trained.items()}
    # Compare successful data events independently of model-specific loss and wall time.
    paired_fields=('sample_ids','augmentation_ids','mask_ids','occurrences')
    def data_trace(rows):
        return [{key:row.get(key) for key in paired_fields} for row in rows]
    paired=data_trace(traces['vit'])==data_trace(traces['fmm'])
    atomic_json(Path(artifact)/'paired_training_summary.json',{'paired_trace_equal':paired,'results':{k:v['resume'] for k,v in trained.items()}})
    if not paired:raise AssertionError('Architectures did not consume paired data/mask traces')
    return trained,diagnostics

def evaluate_four(root,artifact,run_id,config,bundle,trained,initialization):
    import gc
    from fmm_jepa.geometry import clear_geometry_cache
    from fmm_jepa.tiny_imagenet import EvaluationDataset,preprocessing_descriptor
    from fmm_jepa.tiny_evaluation import encoder_state_hash,extract_features,fit_linear_probe,compare_representations,evaluate_heldout
    probes={};features={};performance=[];encoders={}
    for kind in ('vit','fmm'):
        for control in ('untrained','trained'):
            model_id=kind+'_'+control
            clear_geometry_cache();gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
            encoder=copy.deepcopy(trained[kind]['model'].target).cpu()
            if control=='untrained':
                record=initialization['states'][kind]['teacher']
                if digest_file(record['path'])!=record['sha256']:raise ValueError('Initial teacher changed')
                encoder.load_state_dict(torch.load(record['path'],map_location='cpu',weights_only=True))
            encoders[model_id]=encoder
            identity={'dataset_revision':config['data']['source_revision']+'@parquet:'+config['data']['parquet_revision'],
                      'checkpoint_hash':encoder_state_hash(encoder),'encoder_kind':kind,
                      'preprocessing':preprocessing_descriptor(config),'pooling':'mean_final_layernorm_patches',
                      'normalization':'encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization'}
            caches={}
            for split,ids in [('train',bundle['train_ids']),('development',bundle['dev_ids'])]:
                print(f'{model_id}: extracting {len(ids)} {split} features',flush=True)
                caches[split]=extract_features(encoder,EvaluationDataset(bundle['dataset'],ids,config),Path(artifact)/'features'/model_id,
                    {**identity,'sample_ids':list(ids)},batch_size=config['evaluation']['batch_size'],device=config['device'])
                feature_perf=caches[split]['performance']
                if caches[split]['reused']:
                    original=json.loads(Path(caches[split]['path']).with_suffix('.json').read_text())['performance']
                    feature_perf={**original,'historical_measurement':True,'current_invocation':'NOT_RUN_valid_cache_reused'}
                performance.append({**feature_perf,'model_id':model_id,'split':split,'checkpoint_hash':identity['checkpoint_hash'],'seed':config['seeds']['model_init']})
            features[model_id]=caches
            clear_geometry_cache();gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
            probes[model_id]=fit_linear_probe(caches['train'],caches['development'],Path(root)/'checkpoints'/(run_id+'-probes'),config,model_id,encoder=encoder)
            value=probes[model_id]
            write_csv(Path(artifact)/'predictions'/(model_id+'_development.csv'),value['predictions'])
            atomic_json(Path(artifact)/(model_id+'_probe_summary.json'),value['summary'])
            probe_perf_path=Path(artifact)/(model_id+'_probe_performance.json')
            if value['summary']['resumed'] and value['performance'].get('batches_measured',0)==0 and probe_perf_path.exists():
                value['performance']={**json.loads(probe_perf_path.read_text()),'historical_measurement':True,'current_invocation':'NOT_RUN_completed_probe_reused'}
            else:atomic_json(probe_perf_path,value['performance'])
            performance.append({**value['performance'],'model_id':model_id,'split':'train_and_development','checkpoint_hash':identity['checkpoint_hash'],'seed':config['seeds']['probe']})
            print(f'{model_id}: development {value["summary"]["metrics"]}',flush=True)
            gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
    comparison=compare_representations(probes,config)
    if config['evaluation']['heldout_enabled']:
        if config['run_mode']!='pilot':raise ValueError('Held-out must be explicit pilot')
        # This branch is reached only after all four development choices are fixed.
        heldout={}
        atomic_json(Path(artifact)/'heldout_choices_fixed.json',{k:v['summary'] for k,v in probes.items()})
        for model_id,encoder in encoders.items():
            identity={**features[model_id]['train']['identity'],'sample_ids':list(bundle['heldout_ids'])}
            cache=extract_features(encoder,EvaluationDataset(bundle['dataset'],bundle['heldout_ids'],config),Path(artifact)/'features'/model_id,identity,config['evaluation']['batch_size'],config['device'])
            selected={**probes[model_id]['selected_probe'],'choices_fixed':True}
            heldout[model_id]=evaluate_heldout(cache,selected,config,model_id,Path(artifact)/'predictions')
            performance.append({**cache['performance'],'model_id':model_id,'split':'heldout','checkpoint_hash':identity['checkpoint_hash']})
        comparison['heldout_status']='PASS';comparison['heldout_results']={k:v['metrics'] for k,v in heldout.items()}
        atomic_json(Path(artifact)/'heldout_results.json',comparison['heldout_results'])
    else:atomic_json(Path(artifact)/'heldout_results.json',{'status':'NOT_RUN','reason':'Predeclared heldout_enabled=false; no held-out features or predictions computed.'})
    write_csv(Path(artifact)/'probe_metrics.csv',[r for v in probes.values() for r in v['metrics']])
    atomic_json(Path(artifact)/'comparison_summary.json',comparison)
    return probes,comparison,performance


In [ ]:

def finish_run(root,artifact,run_id,config,predecessor,bundle,counts,protocol,certificates,trained,diagnostics,comparison,performance):
    artifact=Path(artifact);root=Path(root)
    for row in performance:
        row.setdefault('run_id',run_id);row.setdefault('seed',config['seeds']['probe'] if row.get('phase')=='probe_fitting' else config['seeds']['model_init'])
        row.setdefault('split','train' if row.get('phase')=='pretraining' else 'see_phase_scope')
        if row.get('model_id') in trained:row.setdefault('checkpoint_manifest',str(Path(trained[row['model_id']]['checkpoint_dir'])/'manifest.json'))
    def passed(value):
        def statuses(item):
            if isinstance(item,dict):return ([item['status']] if 'status' in item else []) + [s for v in item.values() for s in statuses(v)]
            if isinstance(item,list):return [s for v in item for s in statuses(v)]
            return []
        values=statuses(value)
        return bool(values) and all(v=='PASS' for v in values)
    gates={}
    def gate(name,ok,evidence,note):gates[name]={'status':'PASS' if ok else 'FAIL','evidence':str(artifact/evidence),'details':note}
    gate('E1',predecessor['status']=='PASS','predecessor_verification.json','Compatible exported core hashes and fresh compact numerical/model/checkpoint checks.')
    for name in ('E2','E3','E4'):gate(name,passed(certificates['data']),'verification_suite.json','Pinned real dataset, split/subset and deterministic preprocessing certificate.')
    for name in ('E5','E6','E8'):gate(name,passed(certificates['integration']),'verification_suite.json','Real-image isolation, predictor inputs, near/far gradients and EMA checks.')
    finite=all(v['resume']['complete'] and len(v['metrics'])==config['training']['updates'] and all(math.isfinite(r['loss']) and math.isfinite(r['gradient_norm_preclip']) for r in v['metrics']) for v in trained.values())
    gate('E7',finite,'training_metrics.csv','Both registered successful-update budgets completed with finite loss, gradients, activations and post-update parameters.')
    gate('E9',passed(certificates['accounting']),'verification_suite.json','Accumulation, teacher optimizer exclusion and successful-update schedule accounting.')
    gate('E10',passed(certificates['resume']),'resume/resume_certificate.json','CPU FP32 single-thread deterministic4-update trajectories, split after2; fresh-state model/optimizer/scheduler/RNG and data identities exactly match.')
    gate('E11',passed(certificates['probe']) and all(v['encoder_unchanged'] for v in comparison['results'].values()),'comparison_summary.json','Actual four encoder state hashes unchanged during probe fit; cache and normalization unit checks.')
    gate('E12',set(comparison['results'])=={'vit_untrained','vit_trained','fmm_untrained','fmm_trained'} and all(v['completed_epochs']==config['probe']['epochs'] for v in comparison['results'].values()),'comparison_summary.json','Exact own initialization controls, same probe initialization, budget, data and preprocessing.')
    performance_ok=all(any(r.get('model_id')==k and r.get('phase')=='encoder_forward' and r.get('status')=='PASS' and r.get('median_seconds',0)>0 for r in performance) for k in ('vit','fmm'))
    performance_ok &= all(any(r.get('model_id')==k and r.get('phase')=='pretraining' and r.get('updates_measured',0)>0 and r.get('complete_jepa_step_seconds') and r.get('data_preprocessing_seconds') for r in performance) for k in ('vit','fmm'))
    performance_ok &= all(any(r.get('model_id')==k and r.get('phase')==phase and r.get('status')=='PASS' and r.get('batches_measured',0)>0 and r.get('wall_seconds',0)>0 for r in performance) for k in comparison['results'] for phase in ('feature_extraction','probe_fitting'))
    gate('E13',performance_ok,'performance_metrics.json','Measured five separate phases with nonempty finite samples; actual encoder backend profiler; matched full-grid workload. Historical measurements explicitly marked on resume.')
    gates['E14']={'status':'NOT_RUN','evidence':str(artifact/'clean_execution.json'),'details':'External fresh-kernel executor must close this gate after all cells finish.'}
    completed=all(v['resume']['complete'] for v in trained.values());smoke=config['run_mode']=='smoke'
    final_diagnostics=[row for row in diagnostics if row.get('step')==config['training']['updates']]
    degeneracy=len(final_diagnostics)!=2 or any(row.get('zero_variance_flag',False) or row.get('very_low_rank_flag',False) for row in final_diagnostics)
    delta=comparison['deltas']['delta_fmm']['correct_top1']
    positive=not smoke and completed and delta['ci95_percentage_points'][0]>0 and not degeneracy
    comparison['learning_evidence']='inconclusive' if smoke or not completed else ('positive' if positive else ('negative' if delta['ci95_percentage_points'][1]<0 else 'inconclusive'))
    comparison['degeneracy_flags_observed']=degeneracy
    comparison['diagnostic_scope']='200 fixed development images, raw pooled final-LayerNorm features before probe standardization; advisory rank<2 and variance<=1e-12 flags.'
    comparison['ready_for_notebook_02']=False
    learning={
      'L1':{'status':'PASS' if not smoke and completed else 'NOT_RUN','details':'Pilot budget completed.' if not smoke and completed else 'Only integration smoke executed; 5000-update pilot not launched.'},
      'L2':{'status':'PASS','evidence':str(artifact/'comparison_summary.json'),'details':'All four trained/initialized results available; smoke is not learning evidence.' if smoke else 'All four trained/initialized results available at the registered pilot and probe budgets.'},
      'L3':{'status':'PASS','evidence':str(artifact/'representation_diagnostics.json'),'details':'Raw dispersion, norm, cosine, covariance rank, matching-patch variation and branch activity inspected. Flags='+str(degeneracy)},
      'L4':{'status':comparison['heldout_status'],'evidence':str(artifact/'heldout_results.json'),'details':'Explicit held-out evaluation status; source valid excluded from all model/probe choices.'},
      'L5':{'status':'PASS','details':'One paired training seed; bootstrap intervals exclude training-seed variability.'},
      'L6':{'status':'PASS','evidence':str(artifact/'comparison_summary.json'),'details':'Within-model deltas separate from architecture delta; unequal parameter/compute counts reported.'}}
    atomic_json(artifact/'certificate_ledger.json',{'engineering':gates,'learning':learning})
    atomic_json(artifact/'comparison_summary.json',comparison)
    # Preserve nested timing details in JSON and flattened scalar/JSON values in CSV.
    atomic_json(artifact/'performance_metrics.json',performance)
    rows=[{key:json.dumps(value,sort_keys=True) if isinstance(value,(dict,list)) else value for key,value in row.items()} for row in performance]
    write_csv(artifact/'performance_metrics.csv',rows)
    checkpoint_refs={kind:{'directory':value['checkpoint_dir'],'manifest':str(Path(value['checkpoint_dir'])/'manifest.json'),'purpose':'exact resume; final registered-budget EMA teacher for primary evaluation','resume':value['resume']} for kind,value in trained.items()}
    atomic_json(artifact/'checkpoints'/'locations.json',checkpoint_refs)
    handoff={'schema_version':ARTIFACT_SCHEMA,'config_schema_version':CONFIG_SCHEMA,'code_version':WORKFLOW_VERSION,
      'predecessor_run_id':predecessor['predecessor_run_id'],'run_id':run_id,'artifact_directory':str(artifact),
      'notebook':str(root/'notebooks/01_TinyImageNet_Debug.ipynb'),'run_mode':config['run_mode'],'config':config,
      'source_hashes':protocol['source_hashes'],'protocol_hash':identity_hash(protocol),
      'dataset_fingerprint':bundle['dataset'].fingerprint,'split_sha256':bundle['split_manifest']['split_sha256'],
      'dataset_repository':config['data']['repo_id'],'source_revision':config['data']['source_revision'],'parquet_revision':config['data']['parquet_revision'],
      'selected_sizes':bundle['split_manifest']['selected_sizes'],'pool_sizes':bundle['split_manifest']['pool_sizes'],
      'parameter_counts':counts,'checkpoint_locations':checkpoint_refs,'initialization':protocol['initialization'],
      'import_paths':['fmm_jepa.tiny_imagenet','fmm_jepa.tiny_training','fmm_jepa.tiny_evaluation','fmm_jepa.tiny_performance','fmm_jepa.tiny_workflow'],
      'training':{k:{**v['resume'],'attempted_updates':v['metrics'][-1]['attempted_updates'],'clipped_updates':sum(r['clipped'] for r in v['metrics'])} for k,v in trained.items()},
      'probe_results':comparison['results'],'deltas':comparison['deltas'],'heldout_status':comparison['heldout_status'],
      'collapse_diagnostics':diagnostics,'performance':performance,'engineering_ready':False,
      'learning_evidence':comparison['learning_evidence'],'ready_for_notebook_02':False,
      'readiness_reason':'Smoke only; useful FMM learning is unestablished and the explicitly selected real-data pilot remains unexecuted.' if smoke else 'Await external clean-execution gate and registered learning criteria.',
      'readiness_override':None,'essential_engineering_gates':gates,
      'next_step':('Explicitly run Tiny ImageNet pilot before deciding on Food-101. If later authorized and evidence supports it: one bounded same-scaffold Food-101 comparison from fresh initialization, all four controls, separate preregistration; no automatic transfer.' if smoke else 'Review the completed Tiny ImageNet pilot, its own-initialization controls, raw representation diagnostics and measured compute. If the recorded criteria are met and the next stage is separately authorized, preregister one bounded same-scaffold Food-101 comparison from fresh initialization with all four controls. Otherwise investigate the observed limitation before choosing a new experiment; no automatic pilot rerun or dataset transition.'),
      'limitations':[('Single paired seed; short smoke and two probe epochs cannot establish useful learning.' if smoke else 'One paired training seed; development selects probe epochs, so the paired bootstrap is exploratory and excludes training-seed variability.'),'Initial integration run20260923T173233Z-82b71a99 failed when diagnostic inference geometry entered training; preserved evidence and a diagnostic-to-backward regression accompany the cache-boundary fix.','Source valid held out from predictive evaluation unless explicit pilot flag.','Exact resume certificate scoped to CPU FP32 one thread and pinned software; no cross-device bitwise claim.','64-token timings cannot establish asymptotic scaling; p4FMM has16nodes/cell and greater parameter count.','No mixed precision/multiworker/distributed resume validated.','Persistent boot disk; instance deletion policy unverified; verify off-instance best/final hashes before recycle.']}
    atomic_json(artifact/'handoff_01_to_02.json',handoff)
    atomic_json(artifact/'run_manifest.json',{'schema_version':ARTIFACT_SCHEMA,'run_id':run_id,'status':'COMPLETED_AWAITING_EXTERNAL_CLEAN_CERTIFICATE','artifact_directory':str(artifact),'protocol_hash':identity_hash(protocol),'source_hashes':protocol['source_hashes'],'config_hash':identity_hash(config),'checkpoint_locations':checkpoint_refs})
    write_report(root,artifact)
    return handoff

def write_report(root,artifact):
    root=Path(root);artifact=Path(artifact);h=json.loads((artifact/'handoff_01_to_02.json').read_text())
    ledger=json.loads((artifact/'certificate_ledger.json').read_text());cfg=h['config']
    smoke=h['run_mode']=='smoke'
    resume_profile='full_pool' if cfg.get('protocol_variant')=='full_training_pool_explicit_launch_only' else h['run_mode']
    if smoke:
        learning_answer='The smoke budget cannot establish this. Use its own initialization control and a completed pilot before making that claim.'
        architecture_answer='Not established by a one-seed smoke. Quality and measured compute are reported independently.'
        readiness_answer='No. Complete and assess the registered Tiny ImageNet pilot first; no readiness override was made.'
    else:
        learning_answer={'positive':'This pilot supports improvement over FMM initialization under the registered criterion; the conclusion remains conditional on one training seed and development-based probe selection.',
                         'negative':'This pilot shows a decrease relative to FMM initialization under the registered comparison; it does not establish permanent inability to learn.',
                         'inconclusive':'This pilot does not establish useful improvement under the registered criterion. Inspect the initialization contrast and raw diagnostic trajectory before choosing the next experiment.'}.get(h['learning_evidence'],'Inspect the recorded learning status and evidence.')
        architecture_delta=h['deltas']['delta_architecture']['correct_top1']
        lo,hi=architecture_delta['ci95_percentage_points']
        architecture_answer=('FMM has higher development top-1 in this paired pilot.' if lo>0 else 'FMM has lower development top-1 in this paired pilot.' if hi<0 else 'The paired interval does not resolve a development top-1 advantage.')
        architecture_answer+=f" The FMM-minus-ViT difference is {architecture_delta['delta_percentage_points']:+.2f} percentage points, paired 95% interval {architecture_delta['ci95_percentage_points']}. Interpret quality alongside the measured compute; one seed does not establish a robust architectural advantage."
        readiness_answer=('The recorded readiness criteria are met. Review the raw diagnostics and limitations, then separately authorize and preregister a fresh-initialization Food-101 comparison.' if h['ready_for_notebook_02'] else 'No. The recorded criteria are not all met; review the completed pilot evidence and resolve the observed limitations. No readiness override was made.')
    lines=['# Notebook 01 executed evidence','',f"Run `{h['run_id']}`; predecessor `{h['predecessor_run_id']}`. Profile **{h['run_mode']}**.",'',
       f"Engineering ready: **{h['engineering_ready']}**. Learning evidence: **{h['learning_evidence']}**. Ready for Notebook02: **{h['ready_for_notebook_02']}**.",'',h['readiness_reason'],'',
       '## Data and protocol','',f"Intended repository: `{h['dataset_repository']}`; source `{h['source_revision']}`; converted Parquet `{h['parquet_revision']}`.",
       f"Selected memberships: {h['selected_sizes']}. Duplicate-cleaned pools: {h['pool_sizes']}. 27 exact duplicate copies excluded before selection; no relabeling. All 110,000 source images decoded and class mapping verified.",
       f"Native 64×64, patch8,64 tokens. Same-scaffold dim{cfg['model']['dim']},depth{cfg['model']['depth']}, FMMp{cfg['model']['order']},leaf{cfg['model']['leaf_size']}; not parameter matched. Device{cfg['device']},FP32. Equal effectivebatch{cfg['effective_batch_size']}.",'',
       '## Actual model budgets','', '|Model|Trainable parameters|Teacher bytes|Successful updates|Attempts|Image exposures|', '|---|---:|---:|---:|---:|---:|']
    for kind in ('vit','fmm'):
        c=h['parameter_counts'][kind];t=h['training'][kind]
        lines.append(f"|{kind}|{c['trainable']:,}|{c['teacher_state_bytes']:,}|{t['final_step']}|{t['attempted_updates']}|{t['actual_image_exposures']}|")
    lines+=['','## Frozen representations','',f"Complete-image EMA target, final affine LayerNorm patch outputs averaged over64positions; predictor absent. Training-only feature standardization. Linear200-class AdamW, lr{cfg['probe']['lr']},weightdecay{cfg['probe']['weight_decay']},batch{cfg['probe']['batch_size']},{cfg['probe']['epochs']}epochs for every control. Best development top1, earliest tie. Initialization and data order matched.",'', '|Representation|Dev top1|Dev top5|Examples|', '|---|---:|---:|---:|']
    for key,v in h['probe_results'].items():
        m=v['metrics'];lines.append(f"|{key}|{100*m['top1']:.2f}%|{100*m['top5']:.2f}%|{m.get('examples',m.get('n_examples','see summary'))}|")
    lines+=['','|Paired top1 contrast|Delta percentage points|95% interval|','|---|---:|---|']
    for key,v in h['deltas'].items():
        d=v['correct_top1'];lines.append(f"|{key}|{d['delta_percentage_points']:+.2f}|{d['ci95_percentage_points']}|")
    lines+=['','Intervals resample development images conditional on these fixed models. They do not measure variation across training seeds. Uniform0.5% chance is only a sanity reference.',f"Held-out predictive evaluation: **{h['heldout_status']}**. Dataset integrity auditing is separate from predictive evaluation.",'', '## Raw representation diagnostics','', '|Encoder|Step|Mean feature std|Mean cosine|Effective rank|Rank ceiling|Far norm|','|---|---:|---:|---:|---:|---:|---:|']
    for d in h['collapse_diagnostics']:
        lines.append(f"|{d['model_id']}|{d['step']}|{d['feature_std_mean']:.4g}|{d['between_image_cosine_mean']:.4g}|{d['effective_rank']:.3f}|{d['effective_rank_ceiling']}|{d.get('fullimage_far_norm')}|")
    lines+=['','Diagnostics use200fixed development images before probe standardization. Rank<2 and variance≤1e−12 are advisory registered warnings. Nonzero far output/gradient is branch activity, not useful long-range semantics. Full covariance spectra and matching-position across-image dispersion are saved in representation_diagnostics.json.','', '## Measured performance','', 'Full per-phase timings, quartiles, sample counts, peak allocations, backend events and workload definitions are in performance_metrics.json and its CSV. All dedicated encoder measurements use the same complete64-token real-image batch. Training step timing includes data/preprocessing and optimizer/EMA; excludes checkpoint/diagnostic overhead, which is reported in total wall time.','']
    lines+=['|Model|Data/update ms|JEPA update ms|Encoder forward ms|Encoder peak MiB|Training invocation peak MiB|Training wall s|','|---|---:|---:|---:|---:|---:|---:|']
    for kind in ('vit','fmm'):
        t=next(r for r in h['performance'] if r['model_id']==kind and r['phase']=='pretraining')
        e=next(r for r in h['performance'] if r['model_id']==kind and r['phase']=='encoder_forward')
        def number(v,scale=1):return 'NOT_RUN' if v is None else f'{v*scale:.3f}'
        lines.append('|'+kind+'|'+number(t['data_preprocessing_seconds']['median'],1000)+'|'+number(t['complete_jepa_step_seconds']['median'],1000)+'|'+number(e['median_seconds'],1000)+'|'+number(e['peak_allocated_bytes'],1/2**20)+'|'+number(t['peak_cuda_allocated_bytes'],1/2**20)+'|'+number(t['wall_seconds_including_checkpoints_and_diagnostics'])+'|')
    lines+=['','Training invocation peak includes diagnostic allocations and backend workspace; encoder peaks use a separate full-grid measurement. CUDA allocator peaks exclude reserved cache and context memory. Probe fitting runs on CPU; CPU tensor peak memory was not measured.','', '|Representation|Feature extraction wall s (train+dev)|Probe wall s|','|---|---:|---:|']
    for key in h['probe_results']:
        f=[r['wall_seconds'] for r in h['performance'] if r.get('model_id')==key and r['phase']=='feature_extraction']
        q=[r['wall_seconds'] for r in h['performance'] if r.get('model_id')==key and r['phase']=='probe_fitting']
        lines.append(f'|{key}|{sum(f):.3f}|{sum(q):.3f}|')
    lines+=['','## Certificates and execution','', '|Gate|Status|Evidence|','|---|---|---|']
    for key,v in ledger['engineering'].items():lines.append(f"|{key}|{v['status']}|{Path(v['evidence']).name}|")
    lines+=['','Resume: controlled CPU FP32, single thread, deterministic algorithms, K4/J2 on real data; exact model/optimizer/scheduler/RNG/consumed-order/augmentation/mask comparison for both models. An earlier two-thread FMM test had identical losses but parameter differences up to3.28e−7, retained under development_verification/threaded_resume_observation.json. This does not support cross-hardware/software bitwise equivalence.','', '## Answers supported by this run','',
       f"1. Does the real-data pipeline work? {'Yes, all14 engineering gates passed for the executed profile.' if h['engineering_ready'] else 'One or more required engineering gates are pending or failed; inspect the certificate ledger.'}",
       '2. Did FMM learn useful features? '+learning_answer,
       '3. Advantage over ViT? '+architecture_answer,
       '4. Ready for Food-101? '+readiness_answer,'',
       '## Resume and next stage','', 'All executable project source remains in notebooks. The import hook loads only tagged `ahr-export` definition cells and enables ordinary `from fmm_jepa...` imports. The main notebook calls the same reusable runner in both profiles.','']
    if smoke:
        lines+=['The next unexecuted experiment is the registered Tiny ImageNet pilot. Validate its actual batch and checkpoint recovery first. Training expected to exceed 30 minutes must run in detached tmux; record the session and attach/detach instructions. The external executor must close the fresh-kernel certificate after notebook execution.','']
    else:
        lines+=['The registered pilot has been executed. Review its learning contrasts, representation diagnostics and compute before choosing any new experiment; this report does not request an automatic rerun. Long-running continuation belongs in the recorded detached tmux session.','']
    lines+=['Exact resume: set `FMM_JEPA_01_RESUME='+h['run_id']+'` and `FMM_JEPA_01_PROFILE='+resume_profile+'`, then rerun the notebook from its bootstrap cell through the detached executor. It validates protocol/source/config/dataset identities and loads checkpoint manifests; invalid states raise rather than restart. Training/probe checkpoints retain latest two valid plus best under checkpoints/<run_id>-*. Initial encoder snapshots are immutable controls, not resumable checkpoints.','',
       '## Limitations and next stage','']+['- '+x for x in h['limitations']]+['',h['next_step'],'']
    (artifact/'engineering_report.md').write_text('\n'.join(lines))
    (root/'docs/notebook_02_handoff.md').write_text('\n'.join(lines))

def certify_clean_execution(root,artifact,notebook_path,elapsed_seconds):
    root=Path(root);artifact=Path(artifact);notebook_path=Path(notebook_path)
    doc=json.loads(notebook_path.read_text());cells=[c for c in doc['cells'] if c['cell_type']=='code']
    if not cells or any(c.get('execution_count') is None for c in cells):raise ValueError('A code cell was not executed')
    if any(o.get('output_type')=='error' for c in cells for o in c.get('outputs',[])):raise ValueError('Clean run contains an error')
    h=json.loads((artifact/'handoff_01_to_02.json').read_text());protocol=json.loads((artifact/'protocol.json').read_text())
    if definition_hashes(root)!=protocol['source_hashes']:raise ValueError('Export source changed after protocol registration')
    certificate={'status':'PASS','executor':'nbclient; fresh ahr-go kernel; all cells top-to-bottom','code_cells':len(cells),
      'run_id':h['run_id'],'run_mode':h['run_mode'],'elapsed_seconds':elapsed_seconds,
      'notebook_path':str(notebook_path),'executed_notebook_sha256':digest_file(notebook_path),'source_hashes':protocol['source_hashes']}
    atomic_json(artifact/'clean_execution.json',certificate)
    ledger=json.loads((artifact/'certificate_ledger.json').read_text());ledger['engineering']['E14']={'status':'PASS','evidence':str(artifact/'clean_execution.json'),'details':certificate['executor']}
    ready=all(v['status']=='PASS' for v in ledger['engineering'].values())
    h['engineering_ready']=ready;h['essential_engineering_gates']=ledger['engineering']
    h['ready_for_notebook_02']=ready and h['run_mode']=='pilot' and h['learning_evidence']=='positive'
    if h['run_mode']=='pilot':
        h['readiness_reason']=('All engineering gates passed; registered pilot learning criterion '+('met. Review raw diagnostics and limitations before a separately authorized next stage.' if h['ready_for_notebook_02'] else 'not met; inspect the completed pilot evidence before choosing another experiment.')) if ready else 'One or more engineering gates did not pass; inspect the ledger and completed pilot evidence before proceeding.'
    atomic_json(artifact/'certificate_ledger.json',ledger);atomic_json(artifact/'handoff_01_to_02.json',h)
    comparison=json.loads((artifact/'comparison_summary.json').read_text());comparison['ready_for_notebook_02']=h['ready_for_notebook_02'];atomic_json(artifact/'comparison_summary.json',comparison)
    manifest=json.loads((artifact/'run_manifest.json').read_text());manifest['status']='COMPLETED';manifest['clean_execution']=certificate;atomic_json(artifact/'run_manifest.json',manifest)
    atomic_json(root/'artifacts/notebook_01/latest.json',{'run_id':h['run_id'],'artifact_directory':str(artifact),'clean_execution_status':'PASS'})
    write_report(root,artifact)
    return certificate
